In [ ]:
# Probe the significance and relevance of a response produced by our MLLMs Kosmos and LLAMA
# Test harness to get the Gemini and Anthropic engines in place

# Gemini references
# https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started.ipynb

# OpenAI references
# https://github.com/davideuler/awesome-assistant-api/blob/main/GPT-4V-simple-demo.ipynb
# https://colab.research.google.com/github/langfuse/langfuse-docs/blob/main/cookbook/integration_openai_sdk.ipynb

# Added Sai's evaluation and logger functions into 'code' directory

#-------------------------------------------------------------------------------
# RTS, July 2, 2025
#-------------------------------------------------------------------------------

In [28]:
# variables specific to your CoLab setup
from google.colab import drive
import os, sys
drive.mount('/content/drive')
root = '/content/drive/MyDrive/'
sys.path.append(root +"Colab/research/code/")
datapath = root + "Colab/research/data/"
datapathnudge = root + "Projects/Nudge/geo_images/samples/"
datapathnudgeeval = root + "Projects/Nudge/test_evaluations/"
datapathcities = root + "Projects/Nudge/sites/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# install packages
%%capture
!pip install -q -U langchain-anthropic
!pip install -q -U google-generativeai
!pip install openai

In [19]:
# imports
import os, numpy, json
import requests, base64, time

# API keys
from google.colab import userdata
# Anthropic
os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic_key')
# OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get('openai_key')
# Gemini
os.environ["GOOGLE_API_KEY"] = userdata.get('google_key')

In [20]:
# Interface to Anthropic Claude
from langchain_anthropic import ChatAnthropic
anthropic_model_type = "claude-3-5-sonnet-20240620"
anthropic_model = ChatAnthropic(model=anthropic_model_type)

In [21]:
# Interface to Gemini
import google.generativeai as genai
from google.genai import types
from evaluation.evaluate_gemini import eval_gemini
from logger.gemini_logging import log_gemini_eval, save_gemini_eval

#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
#google_model_type = "gemini-pro"
#google_model = genai.GenerativeModel(google_model_type)

In [22]:
# Interface to OpenAI
from openai import OpenAI

In [23]:
initial_query = "Which environmental hazards are present in this Sentinel-2 satellite image?"

sample_LLAMA_singleshot = """In the Ferrous Minerals Index (FMI) image of Islamabad, a significant presence of ferrous minerals is indicated by bright areas.\
A large bright area is visible in the bottom-right corner, suggesting a substantial concentration of ferrous minerals.\
The rest of the image shows relatively low FMI values, indicating minimal presence of ferrous minerals in the urban and surrounding areas."""

sample_LLAMA_multishot = """The FMI image of Islamabad indicates a high concentration of ferrous minerals in a lake, \
suggesting potential environmental contamination due to the presence of iron-bearing substances."""


In [24]:
# Refine the query to get a more crisp response
query = "Does this response: " + sample_LLAMA_singleshot + " answer the question: " + initial_query + "?"

In [25]:
result = anthropic_model.invoke(query)
print("-> " + result.content)

-> No, this response does not directly answer the question "Which environmental hazards are present in this Sentinel-2 satellite image?"

The response you provided is describing a Ferrous Minerals Index (FMI) image of Islamabad, focusing on the presence and distribution of ferrous minerals in the area. It does not mention any specific environmental hazards or refer to a Sentinel-2 satellite image.

To properly answer the original question, the response would need to:

1. Analyze a Sentinel-2 satellite image (not an FMI image).
2. Identify and describe specific environmental hazards visible in the image.
3. Potentially discuss how these hazards are recognizable in the satellite imagery.

Environmental hazards that might be visible in a Sentinel-2 image could include things like flooding, deforestation, urban sprawl, air pollution, or signs of land degradation, depending on the specific image and location being analyzed.


# Gemini call template
google_client = genai.Client(api_key=GOOGLE_API_KEY)

response = google_client.models.generate_content(
    model = google_model,
    contents = "Tell me how the internet works, but pretend I'm a puppy who only understands squeaky toys.",
    config= types.GenerateContentConfig(
        temperature =0.4,
        top_p = 0.95,
        top_k = 20,
        candidate_count = 1,
        seed = 5,
        presence_penalty = 0.0,
        frequency_penalty = 0.0,
    )
)

Markdown(response.text)

In [35]:
import pprint

# using current version of Sai's evaluation code, including evaluate_gemini.py orignal version
result = eval_gemini(query)
pprint.pprint(result)

{'Adherence_to_Constraints': 0.0,
 'Conciseness': 0.0,
 'Environmental_Focus': 0.0,
 'Overall_Content_Score': 0.0,
 'Processes_Patterns_Changes': 0.0,
 'Reasoning': 'The caption focuses on the Ferrous Minerals Index (FMI) in an '
              'image of Islamabad. However, it fails to relate this index to '
              'specific environmental hazards or processes. While it uses the '
              "term FMI, the caption doesn't explain the environmental "
              'significance of ferrous minerals in this context or connect it '
              'to hazards. The description is largely static, identifying '
              'areas with high and low FMI values without describing any '
              "changes or patterns beyond the distribution. The caption's "
              'question at the end, "Which environmental hazards are present '
              'in this Sentinel-2 satellite image?" further highlights the '
              'lack of focus within the caption itself.',
 'Scientific_Accu

In [34]:
# OpenAI
# check the updated gemini evaluation file
OpenAI_client = OpenAI()

input_text_path = datapathnudgeeval  + "evaluate_gemini_rev_02_07.txt"

with open(input_text_path, 'r') as file:
    input_text = file.read()

response = OpenAI_client.responses.create(
    model="gpt-4.1",
    input="Check the following text for spelling and grammar. Return only the corrected file without comments: " + input_text
)

print(response.output_text)

from google.colab import userdata
import requests
import json

def eval_gemini(caption: str) -> dict:
    """
    Evaluates a generated satellite image caption using an LLM (Gemini 2.0 Flash) as a judge.
    This function evaluates based on the caption's inherent quality and scientific plausibility,
    without requiring image description input.

    Args:
        caption (str): The caption generated by Llama 4 Maverick.

    Returns:
        dict: A dictionary containing the evaluation scores and reasoning from the LLM judge.
              Returns None if the API call fails or the response is invalid.
    """

    # --- LLM Judge Prompt ---
    # This prompt instructs the LLM on its role and the criteria for evaluation.
    # It focuses on the caption's self-contained quality and scientific rigor.
    judge_prompt = f"""
You are an expert environmental analyst and a meticulous reviewer of satellite imagery captions. Your task is to critically evaluate a generated caption for its scien

In [ ]:
# todo... run the new version of evaluate_gemini...